In [1]:
# 1. Load Dataset and Standardize Features
import numpy as np
import pandas as pd
import networkx as nx

from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import adjusted_rand_score, silhouette_score
from scipy.spatial.distance import pdist, squareform
from scipy.sparse.csgraph import minimum_spanning_tree

data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

X = X.iloc[:120]
y = y[:120]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Dataset shape:", X_scaled.shape)

# 2. Construct Distance Matrix and Compute MST
distance_matrix = squareform(pdist(X_scaled, metric='euclidean'))
mst = minimum_spanning_tree(distance_matrix)

G = nx.from_scipy_sparse_array(mst)

print("Distance Matrix Shape:", distance_matrix.shape)
print("Number of MST edges:", mst.nnz)

# 3. Create Two Clusters by Removing the Largest Edge
largest_edge = max(
    G.edges(data=True),
    key=lambda edge: edge[2]['weight']
)

print("Largest edge:", largest_edge[0], "-", largest_edge[1])
print("Largest edge weight:", round(largest_edge[2]['weight'], 4))

G.remove_edge(largest_edge[0], largest_edge[1])
components = list(nx.connected_components(G))

if len(components) != 2:
    raise ValueError(f"Expected 2 clusters, found {len(components)}")

cluster_labels = np.zeros(len(X_scaled), dtype=int)

for cluster_id, component in enumerate(components):
    for node in component:
        cluster_labels[node] = cluster_id

print("Number of clusters:", len(components))
print("Cluster 0 size:", np.sum(cluster_labels == 0))
print("Cluster 1 size:", np.sum(cluster_labels == 1))

# 4. Evaluate Clustering Performance
ari = adjusted_rand_score(y, cluster_labels)
sil_score = silhouette_score(X_scaled, cluster_labels)

print("Adjusted Rand Index (ARI):", round(ari, 4))
print("Silhouette Score:", round(sil_score, 4))


Dataset shape: (120, 30)
Distance Matrix Shape: (120, 120)
Number of MST edges: 119
Largest edge: 68 - 112
Largest edge weight: 8.375
Number of clusters: 2
Cluster 0 size: 119
Cluster 1 size: 1
Adjusted Rand Index (ARI): 0.0075
Silhouette Score: 0.436
